# 06 — Data Quality

Dataset coverage and anomalies that affect interpretation of the reports.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))

In [ ]:
# Inclusive reporting period, based on task start date.
START_DATE = "2024-01-01"
END_DATE = "9999-12-31"
REPORT_DATE = date.today()

query = construct_query("task-history.sql", {"START-DATE": START_DATE, "END-DATE": END_DATE})
history = normalise_history(query_data(query))
period_label = f"{START_DATE} to {END_DATE} ({len(history):,} tasks)"
print(f"Reporting period: {period_label}")


## Full-history coverage

In [ ]:
coverage_query = construct_query("data-coverage.sql", {})
coverage = query_data(coverage_query)
coverage["Missing Location Percentage"] = coverage["Tasks Without Location"].div(coverage["Total Tasks"].replace(0, pd.NA)).mul(100).fillna(0).round(1)
coverage

## Represented reference values

In [ ]:
represented_types = history[["Category", "Task Type"]].drop_duplicates().sort_values(["Category", "Task Type"])
represented_categories = history[["Category"]].drop_duplicates().sort_values("Category")
represented_types

## Reference mapping checks

In [ ]:
# Inner joins in task-history.sql should make these zero; non-zero values reveal damaged or externally modified data.
with pd.option_context("display.max_rows", None):
    unmapped = pd.DataFrame([{"Check": "Blank category names", "Count": int(history["Category"].isna().sum())}, {"Check": "Blank task-type names", "Count": int(history["Task Type"].isna().sum())}, {"Check": "Completion before start", "Count": int((history["Completion Date"] < history["Start Date"]).sum())}])
unmapped

In [ ]:
EXPORT_NAME = "06-data-quality.xlsx"
EXPORT_DATA = {"Coverage": coverage, "Categories": represented_categories, "Task Types": represented_types, "Mapping Checks": unmapped}
export_to_spreadsheet(
    get_export_folder_path(), EXPORT_NAME, EXPORT_DATA
)
print(f"Exported {EXPORT_NAME} to {get_export_folder_path()}")
